In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/features_2000A.csv")

In [5]:
mapping = {
    "home_win": 0,
    "draw": 1,
    "away_win": 2
}

df["target"] = df["result"].map(mapping)

In [6]:
df.head()

,form_diff,goal_diff_diff,scored_diff,conceded_diff,elo_diff,result,target
0,0.200000,0.4,-0.2,-0.6,10.353400,home_win,0
1,0.133333,-0.4,-0.6,-0.2,19.145917,away_win,2
2,0.066667,1.0,0.4,-0.6,-1.107338,home_win,0
3,0.200000,1.0,0.4,-0.6,18.005513,draw,1
4,0.200000,0.2,0.2,0.0,19.297593,draw,1


In [7]:
X = df[
    [
        "form_diff",
        "goal_diff_diff",
        "scored_diff",
        "conceded_diff",
        "elo_diff"
    ]
]

y = df["target"]

In [8]:
from sklearn.model_selection import train_test_split

X_train ,X_test ,y_train ,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['form_diff','goal_diff_diff','scored_diff','conceded_diff','elo_diff']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [10]:
pred = model.predict(X_test)

In [11]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_test,pred))

0.5788819875776398


In [12]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softmax",
    num_class=3,

    n_estimators=300,
    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42
)

xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import loa

In [13]:
preds = xgb.predict(X_test)

accuracy = accuracy_score(y_test,preds)

print(accuracy)

0.5803312629399586


In [14]:
importance = pd.DataFrame({

    "feature": X.columns,

    "importance":
        xgb.feature_importances_

})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
4,elo_diff,0.579575
1,goal_diff_diff,0.148545
3,conceded_diff,0.120097
0,form_diff,0.079214
2,scored_diff,0.072569


In [15]:
df = pd.read_csv("../data/features_2000B.csv")

In [16]:
mapping = {
    "home_win": 0,
    "draw": 1,
    "away_win": 2
}

df["target"] = (
    df["result"]
    .map(mapping)
)

In [17]:
df.head()

,form_diff,goal_diff_diff,scored_diff,conceded_diff,elo_diff,neutral,year,result,tournament_type_continental_qualifier,tournament_type_friendly,tournament_type_nations_league,tournament_type_other,tournament_type_regional,tournament_type_world_cup,tournament_type_world_cup_qualifier,target
0,0.200000,0.4,-0.2,-0.6,10.353400,1,2000,home_win,0,0,0,0,0,0,0,0
1,0.133333,-0.4,-0.6,-0.2,19.145917,1,2000,away_win,0,0,0,0,0,0,0,2
2,0.066667,1.0,0.4,-0.6,-1.107338,1,2000,home_win,0,0,0,0,0,0,0,0
3,0.200000,1.0,0.4,-0.6,18.005513,1,2000,draw,0,0,0,0,0,0,0,1
4,0.200000,0.2,0.2,0.0,19.297593,0,2000,draw,0,0,0,0,0,0,0,1


In [18]:
feature_columns = [

    "form_diff",
    "goal_diff_diff",
    "scored_diff",
    "conceded_diff",

    "elo_diff",

    "neutral",

    "tournament_type_continental_qualifier",
    "tournament_type_friendly",
    "tournament_type_nations_league",
    "tournament_type_other",
    "tournament_type_regional",
    "tournament_type_world_cup",
    "tournament_type_world_cup_qualifier"
]

In [19]:
X = df[feature_columns]

y = df["target"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

In [21]:
xgb.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import loa

In [22]:
preds = xgb.predict(X_test)

In [23]:
accuracy = accuracy_score(
    y_test,
    preds
)

print(
    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.5853


In [24]:
from sklearn.metrics import classification_report, confusion_matrix
print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.61      0.85      0.71      2314
           1       0.30      0.02      0.04      1132
           2       0.55      0.60      0.57      1384

    accuracy                           0.59      4830
   macro avg       0.49      0.49      0.44      4830
weighted avg       0.52      0.59      0.51      4830



In [25]:
cm = confusion_matrix(
    y_test,
    preds
)

print(cm)

[[1977   31  306]
 [ 741   23  368]
 [ 534   23  827]]


In [26]:
importance = pd.DataFrame({

    "feature":
        feature_columns,

    "importance":
        xgb.feature_importances_

})

In [27]:
importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
4,elo_diff,0.294409
5,neutral,0.117938
1,goal_diff_diff,0.091995
7,tournament_type_friendly,0.076199
3,conceded_diff,0.069855
12,tournament_type_world_cup_qualifier,0.055466
6,tournament_type_continental_qualifier,0.048058
11,tournament_type_world_cup,0.043319
0,form_diff,0.043156
9,tournament_type_other,0.042347


In [28]:
df["year"].describe()

count    24147.000000
mean      2013.105934
std          7.455012
min       2000.000000
25%       2007.000000
50%       2013.000000
75%       2019.000000
max       2026.000000
Name: year, dtype: float64

In [29]:
df["year"].value_counts().sort_index()

year
2000     441
2001     944
2002     738
2003     895
2004    1071
2005     787
2006     810
2007     970
2008    1085
2009     918
2010     848
2011    1108
2012     999
2013     950
2014     829
2015    1017
2016     893
2017     919
2018     905
2019    1134
2020     347
2021    1115
2022     960
2023    1047
2024    1231
2025     997
2026     189
Name: count, dtype: int64

In [30]:
train_df = df[
    df["year"] < 2022
]

test_df = df[
    df["year"] >= 2022
]

In [31]:
print(train_df.shape)
print(test_df.shape)

(19723, 16)
(4424, 16)


In [32]:
feature_columns = [

    "form_diff",
    "goal_diff_diff",
    "scored_diff",
    "conceded_diff",

    "elo_diff",

    "neutral",

    "tournament_type_continental_qualifier",
    "tournament_type_friendly",
    "tournament_type_nations_league",
    "tournament_type_other",
    "tournament_type_regional",
    "tournament_type_world_cup",
    "tournament_type_world_cup_qualifier"
]

In [33]:
X_train = train_df[
    feature_columns
]

y_train = train_df[
    "target"
]

X_test = test_df[
    feature_columns
]

y_test = test_df[
    "target"
]

In [34]:
xgb.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import loa

In [35]:
from sklearn.metrics import accuracy_score

preds = xgb.predict(
    X_test
)

accuracy = accuracy_score(
    y_test,
    preds
)

print(
    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.5997


In [36]:
train_df["year"].min()

np.int64(2000)

In [37]:
train_df["year"].max()

np.int64(2021)

In [38]:
print(train_df["year"].min())
print(train_df["year"].max())

print(test_df["year"].min())
print(test_df["year"].max())

print(train_df.shape)
print(test_df.shape)

2000
2021
2022
2026
(19723, 16)
(4424, 16)


In [39]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.63      0.85      0.72      2107
           1       0.38      0.01      0.02      1016
           2       0.55      0.65      0.60      1301

    accuracy                           0.60      4424
   macro avg       0.52      0.51      0.45      4424
weighted avg       0.55      0.60      0.53      4424



In [40]:
from sklearn.metrics import confusion_matrix

print(
    confusion_matrix(
        y_test,
        preds
    )
)

[[1791   10  306]
 [ 618   13  385]
 [ 441   11  849]]


In [41]:
y_test.value_counts(
    normalize=True
)

target
0    0.476266
2    0.294078
1    0.229656
Name: proportion, dtype: float64

In [42]:
import joblib

joblib.dump(
    xgb,
    "../models/world_cup_predictor.pkl"
)

['../models/world_cup_predictor.pkl']